# v12 1D-TCN(seq2seq)で TVT を予測 ― 5-fold CV

v10 の `artifacts_v10/data/train.parquet` を **坑井ごとの系列**に組み直し、双方向(non-causal)
dilated 1D-CNN(TCN)で各MD位置の `target = TVT - last_tvt` を一括予測する(系列→系列回帰)。

狙い: 順方向PF + 逆方向PF を入力チャンネルに与え、CNN に「どこで順方向/逆方向を信じるか」を
局所的に学習させる(= 学習可能な forward-backward 平滑化)。LightGBM が苦手な連続性・局所文脈を補う。

評価: **GroupKFold(5) by well**、OOF 全行プールの **overall RMSE**(Kaggle指標に整合)。
`target` は坑井定数オフセットを引いた相対値なので target上RMSE = TVT上RMSE。

メモ: 773坑井と小さめなので過学習注意(dropout / early stopping / well単位CV)。
入力は parquet の数値特徴を全チャンネル使用(`bpf_*`/`cmp_*` 含む)。

In [1]:
import time, copy, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

In [2]:
# ============================== CONFIG ==============================
PROJECT_DIR = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
PARQUET     = "C:\\Users\\kosaka256\\Documents\\rogii_claude\\v10\\artifacts_v10\\data\\train.parquet"
OUT_DIR     = PROJECT_DIR / "artifacts_v12"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
SEED     = 0
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# TCN
HIDDEN   = 96
LEVELS   = 7        # dilation 1,2,4,...,64 -> 受容野は十分広い(双方向)
KERNEL   = 3
DROPOUT  = 0.10

# 学習
EPOCHS   = 40
BATCH    = 8        # 1バッチの坑井数
LR       = 1e-3
WD       = 1e-5
PATIENCE = 8        # early stopping
GRAD_CLIP= 1.0

torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")

device: cuda NVIDIA GeForce RTX 5070


In [5]:
import pyarrow.parquet as pq

# --- メタ列だけ先に読む(小さい)---
NON_FEAT = {"well", "id", "target"}
all_cols = pq.read_schema(PARQUET).names
feat_cols = [c for c in all_cols if c not in NON_FEAT]   # 残りは数値特徴
F = len(feat_cols)
meta = pq.read_table(PARQUET, columns=["well", "id", "target"]).to_pandas()
n = len(meta)
print(f"rows={n}  wells={meta['well'].nunique()}  features={F}")

# --- 坑井が連続ブロックである前提を検証(ビュー化のため)---
wcol = meta["well"].to_numpy()
bounds = np.concatenate([[0], np.flatnonzero(wcol[1:] != wcol[:-1]) + 1, [n]])
well_order = wcol[bounds[:-1]]
assert len(set(well_order)) == len(well_order), "wellが連続でない→要ソート(別途対応)"

# --- Xall を事前確保し、列チャンクで直接書き込む(ピーク低)---
Xall = np.empty((n, F), np.float32)
CHUNK = 64
for i in range(0, F, CHUNK):
    cols = feat_cols[i:i+CHUNK]
    sub = np.array(pq.read_table(PARQUET, columns=cols).to_pandas().to_numpy(np.float32))  # 書込可コピー
    sub[~np.isfinite(sub)] = np.nan
    Xall[:, i:i+len(cols)] = sub
    del sub
yall = meta["target"].to_numpy(np.float32)

# --- store はスライス(ビュー)。コピーしない ---
store = {}; wells = []
for b in range(len(well_order)):
    s, e = int(bounds[b]), int(bounds[b+1])
    w = well_order[b]
    store[w] = (Xall[s:e], yall[s:e], np.arange(s, e))   # Xall[s:e] はビュー
    wells.append(w)
wells = np.array(wells)
print("n wells:", len(wells), " T min/med/max =",
      int(np.min([e-s for s,e in zip(bounds[:-1],bounds[1:])])),
      int(np.median([e-s for s,e in zip(bounds[:-1],bounds[1:])])),
      int(np.max([e-s for s,e in zip(bounds[:-1],bounds[1:])])))

rows=3783989  wells=773  features=655
n wells: 773  T min/med/max = 407 4840 10052


In [6]:
# 標準化スケーラ(train坑井から running sum で算出。大きな concat を避ける)
def compute_scaler(train_wells, store, F):
    s = np.zeros(F, np.float64); s2 = np.zeros(F, np.float64); n = np.zeros(F, np.float64)
    for w in train_wells:
        X = store[w][0]
        m = np.isfinite(X)
        s  += np.where(m, X, 0).sum(0)
        s2 += np.where(m, X.astype(np.float64)**2, 0).sum(0)
        n  += m.sum(0)
    n = np.maximum(n, 1)
    mean = s / n
    var = np.maximum(s2 / n - mean**2, 0.0)
    std = np.sqrt(var); std[std < 1e-6] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)

In [7]:
# Dataset / collate(系列をパディング + マスク)
class WellDS(Dataset):
    def __init__(self, well_list, store, mean, std):
        self.wells = list(well_list); self.store = store
        self.mean = mean; self.std = std
    def __len__(self): return len(self.wells)
    def __getitem__(self, i):
        X, y, _ = self.store[self.wells[i]]
        Xn = (X - self.mean) / self.std
        Xn = np.nan_to_num(Xn, nan=0.0, posinf=0.0, neginf=0.0)
        return torch.from_numpy(Xn), torch.from_numpy(y)

def collate(batch):
    Tmax = max(x.shape[0] for x, _ in batch)
    B = len(batch); F = batch[0][0].shape[1]
    X = torch.zeros(B, Tmax, F); Y = torch.zeros(B, Tmax); M = torch.zeros(B, Tmax)
    for i, (x, y) in enumerate(batch):
        t = x.shape[0]; X[i, :t] = x; Y[i, :t] = y; M[i, :t] = 1.0
    return X.transpose(1, 2).contiguous(), Y, M   # X: (B,F,T)

In [8]:
# 双方向(non-causal)dilated TCN
class TCNBlock(nn.Module):
    def __init__(self, c_in, c_out, k, d, dropout):
        super().__init__()
        pad = d * (k - 1) // 2            # same-length(両側パディング=双方向)
        self.conv1 = nn.Conv1d(c_in, c_out, k, padding=pad, dilation=d)
        self.conv2 = nn.Conv1d(c_out, c_out, k, padding=pad, dilation=d)
        self.drop = nn.Dropout(dropout)
        self.act = nn.GELU()
        self.down = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else None
    def forward(self, x):
        res = x if self.down is None else self.down(x)
        y = self.drop(self.act(self.conv1(x)))
        y = self.drop(self.act(self.conv2(y)))
        return self.act(y + res)

class TCN(nn.Module):
    def __init__(self, F, H, levels, k, dropout):
        super().__init__()
        self.inp = nn.Conv1d(F, H, 1)
        self.tcn = nn.Sequential(*[TCNBlock(H, H, k, 2**i, dropout) for i in range(levels)])
        self.head = nn.Conv1d(H, 1, 1)
    def forward(self, x):                 # x: (B,F,T)
        return self.head(self.tcn(self.inp(x))).squeeze(1)   # (B,T)

In [9]:
# 予測 / 評価ユーティリティ(val/predict は1坑井ずつ、パディング無し)
def predict_well(model, w, store, mean, std, device):
    X = store[w][0]
    Xn = np.nan_to_num((X - mean) / std, nan=0.0, posinf=0.0, neginf=0.0)
    xt = torch.from_numpy(Xn).transpose(0, 1).unsqueeze(0).to(device)   # (1,F,T)
    model.eval()
    with torch.no_grad():
        return model(xt).squeeze(0).float().cpu().numpy()

def eval_rmse(model, wells_, store, mean, std, device):
    se = 0.0; n = 0
    for w in wells_:
        p = predict_well(model, w, store, mean, std, device)
        y = store[w][1]
        se += float(np.sum((p - y) ** 2)); n += len(y)
    return math.sqrt(se / max(n, 1))

In [10]:
def run_fold(tr_wells, va_wells, store, F, device, verbose=True):
    mean, std = compute_scaler(tr_wells, store, F)
    ds = WellDS(tr_wells, store, mean, std)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=0)
    model = TCN(F, HIDDEN, LEVELS, KERNEL, DROPOUT).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    best = float("inf"); best_state = None; patience = 0
    for ep in range(EPOCHS):
        model.train()
        for X, Y, M in loader:
            X, Y, M = X.to(device), Y.to(device), M.to(device)
            pred = model(X)
            loss = ((pred - Y) ** 2 * M).sum() / M.sum().clamp_min(1.0)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        vr = eval_rmse(model, va_wells, store, mean, std, device)
        if vr < best - 1e-4:
            best = vr; best_state = copy.deepcopy(model.state_dict()); patience = 0
        else:
            patience += 1
        if verbose:
            print(f"    epoch {ep:2d}: val_rmse={vr:.4f}  (best={best:.4f})")
        if patience >= PATIENCE:
            if verbose: print(f"    early stop @ epoch {ep}")
            break
    model.load_state_dict(best_state)
    preds = {w: predict_well(model, w, store, mean, std, device) for w in va_wells}
    return preds, best

In [11]:
# 5-fold GroupKFold by well
gkf = GroupKFold(n_splits=N_SPLITS)
well_groups = np.arange(len(wells))   # split は wells 単位
oof = np.full(len(df), np.nan, np.float32)
fold_rmse = []
t0 = time.perf_counter()
for k, (tr_idx, va_idx) in enumerate(gkf.split(wells, groups=wells)):
    print(f"\n===== fold {k} (train wells={len(tr_idx)}, val wells={len(va_idx)}) =====")
    tr_wells = wells[tr_idx]; va_wells = wells[va_idx]
    preds, best = run_fold(tr_wells, va_wells, store, F, DEVICE)
    fold_rmse.append(best)
    for w in va_wells:
        oof[store[w][2]] = preds[w].astype(np.float32)
    print(f"  fold {k} best val RMSE = {best:.4f}")
print(f"\nfolds done in {(time.perf_counter()-t0)/60:.1f} min")


===== fold 0 (train wells=618, val wells=155) =====


C:\Users\kosaka256\AppData\Local\Temp\ipykernel_6188\3984252350.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  return torch.from_numpy(Xn), torch.from_numpy(y)


    epoch  0: val_rmse=8.6254  (best=8.6254)
    epoch  1: val_rmse=9.0761  (best=8.6254)
    epoch  2: val_rmse=8.2207  (best=8.2207)
    epoch  3: val_rmse=8.5158  (best=8.2207)
    epoch  4: val_rmse=9.0799  (best=8.2207)
    epoch  5: val_rmse=8.6028  (best=8.2207)
    epoch  6: val_rmse=9.0929  (best=8.2207)
    epoch  7: val_rmse=7.8321  (best=7.8321)
    epoch  8: val_rmse=7.8551  (best=7.8321)
    epoch  9: val_rmse=7.9600  (best=7.8321)
    epoch 10: val_rmse=8.5077  (best=7.8321)
    epoch 11: val_rmse=8.0141  (best=7.8321)
    epoch 12: val_rmse=8.3773  (best=7.8321)
    epoch 13: val_rmse=7.8080  (best=7.8080)
    epoch 14: val_rmse=8.3979  (best=7.8080)
    epoch 15: val_rmse=8.0195  (best=7.8080)
    epoch 16: val_rmse=8.3763  (best=7.8080)
    epoch 17: val_rmse=8.1844  (best=7.8080)
    epoch 18: val_rmse=8.2825  (best=7.8080)
    epoch 19: val_rmse=8.6197  (best=7.8080)
    epoch 20: val_rmse=8.1661  (best=7.8080)
    epoch 21: val_rmse=8.3794  (best=7.8080)
    early 

In [12]:
# 結果(overall RMSE)
mask = np.isfinite(oof)
overall = float(np.sqrt(np.mean((oof[mask] - yall[mask]) ** 2)))
tmp = pd.DataFrame({"well": df["well"].to_numpy(), "e2": (oof - yall) ** 2})
per_well = float(np.sqrt(tmp.groupby("well")["e2"].mean()).mean())
np.save(OUT_DIR / "oof_tcn.npy", oof)

print("fold RMSE:", [round(r, 4) for r in fold_rmse])
print(f"mean fold RMSE   = {np.mean(fold_rmse):.4f}")
print(f"OVERALL RMSE     = {overall:.4f}   (= Kaggle指標に整合, 全OOF行プール)")
print(f"per-well mean RMSE = {per_well:.4f}  (参考)")
print(f"oof saved -> {OUT_DIR/'oof_tcn.npy'}")

fold RMSE: [7.808, 8.1609, 6.7822, 7.3309, 9.0863]
mean fold RMSE   = 7.8337
OVERALL RMSE     = 7.8710   (= Kaggle指標に整合, 全OOF行プール)
per-well mean RMSE = 6.3295  (参考)
oof saved -> C:\Users\kosaka256\Documents\rogii_claude\artifacts_v12\oof_tcn.npy
